# 01 — EDA (Exploratory Data Analysis)

Цей зошит виконує розвідковий аналіз даних для системи виявлення шахрайства.

**Цілі:**
1. Завантажити навчальні дані з SQLite бази.
2. Проаналізувати базову інформацію та пропуски.
3. Дісбаланс класів.
4. Аналіз ключових розподілів між легітимними та шахрайськими класами.
5. Топ-сигнали шахрайства.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

## 1. Завантаження даних з SQLite
Завантажуємо таблиці `users` та `transactions` з бази `data/sql/train_data.db`.

In [ ]:
conn = sqlite3.connect('../data/sql/train_data.db')

df_users = pd.read_sql('SELECT * FROM users', conn)
df_transactions = pd.read_sql('SELECT * FROM transactions', conn)

conn.close()

print(f"Розмір таблиці Users: {df_users.shape}")
print(f"Розмір таблиці Transactions: {df_transactions.shape}")

## 2. Базова інформація та пропуски

In [ ]:
print("=== Інформація по датасету Users ===")
df_users.info()
print("\nПропуски в таблиці Users:")
print(df_users.isnull().sum()[df_users.isnull().sum() > 0])

print("\n" + "="*50 + "\n")

print("=== Інформація по датасету Transactions ===")
df_transactions.info()
print("\nПропуски в таблиці Transactions:")
print(df_transactions.isnull().sum()[df_transactions.isnull().sum() > 0])

## 3. Дисбаланс класів (`is_fraud`)

In [ ]:
fraud_counts = df_users['is_fraud'].value_counts()
fraud_pct = df_users['is_fraud'].value_counts(normalize=True) * 100

balance_df = pd.DataFrame({
    'Кількість': fraud_counts,
    'Відсоток %': fraud_pct.round(2)
})
display(balance_df)

sns.countplot(data=df_users, x='is_fraud', hue='is_fraud', palette='Set2', legend=False)
plt.title('Дисбаланс класів (is_fraud)')
plt.xlabel('Клас (0 - Legitimate, 1 - Fraud)')
plt.ylabel('Кількість користувачів')
plt.show()

## 4. Аналіз ключових розподілів
Merger транзакцій з користувачами для аналізу поведінки за класом `is_fraud`.

In [ ]:
df_merged = df_transactions.merge(df_users[['id_user', 'is_fraud']], on='id_user', how='inner')
df_merged['timestamp_tr'] = pd.to_datetime(df_merged['timestamp_tr'], errors='coerce')
print(f"Merged shape: {df_merged.shape}")

### 4.1 Розподіл сум транзакцій за класами

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_merged, x='is_fraud', y='amount', ax=ax[0])
ax[0].set_title('Суми транзакцій за класами')

df_merged['amount_log1p'] = np.log1p(df_merged['amount'])
sns.kdeplot(data=df_merged, x='amount_log1p', hue='is_fraud', common_norm=False, fill=True, ax=ax[1])
ax[1].set_title('Логарифмічний розподіл сум за класами')
plt.show()

### 4.2 Кількість карток та error_group за класами

In [ ]:
# Кількість унікальних карток на юзера
user_cards = df_merged.groupby(['id_user', 'is_fraud'])['card_mask_hash'].nunique().reset_index(name='n_cards')
print("Медіанна кількість карток:")
display(user_cards.groupby('is_fraud')['n_cards'].median())

# Error groups analysis
print("\nError groups за класами:")
error_fraud = df_merged.groupby(['is_fraud', 'error_group']).size().unstack(fill_value=0)
display(error_fraud)

### 4.3 Traffic type за класами

In [ ]:
traffic_fraud = df_users.groupby('traffic_type').agg(
    fraud_cnt=('is_fraud', 'sum'),
    total=('is_fraud', 'count'),
).assign(fraud_pct=lambda x: (x.fraud_cnt / x.total * 100).round(2))
display(traffic_fraud.sort_values('fraud_pct', ascending=False))

## 5. Ключові сигнали шахрайства

1. **Кількість карток** — шахраї використовують ~3.8x більше карток.
2. **Traffic type organic** — fraud rate 19% (vs 3-4% для інших).
3. **Error group fraud/antifraud** — найсильніші сигнали.
4. **Суми транзакцій** — шахраї мають вищий avg amount.
5. **Success rate** — різко відрізняється між класами.